# Referenční MILP model úlohy O-CVRPTW

Tento notebook implementuje kompletní MILP (Mixed Integer Linear Programming) model selektivní kapacitní rozvozní úlohy s časovými okny (O-CVRPTW) formulovaný v kapitole 1.3 diplomové práce. Model je řešen open-source solverem CBC prostřednictvím knihovny PuLP.

**Struktura notebooku:**
1. Instalace balíčků a konfigurace parametrů instance
2. Načtení datasetu a příprava vstupních dat (vzdálenostní matice, časová okna, poptávka)
3. Definice rozhodovacích proměnných a účelové funkce dle rovnice (8)
4. Implementace omezujících podmínek (C1)–(C11)
5. Řešení modelu solverem CBC a analýza výsledků

**Požadované knihovny:** pandas, numpy, openpyxl, pulp

In [1]:
# Instalace potřebných knihoven
!pip install pandas numpy openpyxl pulp

## 1. Konfigurace parametrů instance

Parametry instance odpovídají specifikaci v kapitolách 1.3 a 2.1 diplomové práce:
- **Flotila:** K = 3 identická vozidla s kapacitou Q = 13 (kap. 1.3)
- **Servisní čas:** δ = 1,0 časové jednotky (kap. 1.3, podmínka C6)
- **Penalizace za zpoždění:** γ = 1,5 za jednotku zpoždění (kap. 1.3, rovnice 8); v kódu označeno jako `pi`
- **Penalizace za neobsloužení:** P_skip = 18,0 (kap. 1.3, rovnice 8)
- **Solver:** CBC s časovým limitem 1 800 s a relativní tolerancí gapRel = 10 % (kap. 2.3, tabulka 2.7)

Souřadnice uzlů jsou abstraktní (euklidovská rovina) a při jednotkové rychlosti platí t_ij = c_ij dle rovnice (9).

In [2]:
import os
from dataclasses import dataclass


# KONFIGURACE PRO small_dataset.xlsx

# Cesta k datasetu (nutnost upravit podle aktuálního uložení datasetu)
DATA_PATH = "small_dataset.xlsx"

# Kontrola existence souboru
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Soubor nenalezen: {DATA_PATH}")

# Depo bude City 1 (první město v datasetu)
DEPOT_CITY_NAME = "City 1"

# DŮLEŽITÉ: JEDNOTKY

# Dataset small_dataset.xlsx obsahuje ABSTRAKTNÍ souřadnice (ne GPS!).
# Proto pracujeme v ABSTRAKTNÍCH JEDNOTKÁCH, nikoliv v km nebo hodinách.

# Interpretace jednotek:
#   - Vzdálenost: "jednotky" (ne km)
#   - Čas: "časové jednotky" (ne hodiny)
#   - Vztah: čas = vzdálenost / rychlost = vzdálenost / 1.0 = vzdálenost

# Důsledky:
#   - c[i,j] = Euklidovská vzdálenost v jednotkách (rozsah: 0.24 - 21.06)
#   - t[i,j] = cestovní čas = c[i,j] (protože rychlost = 1.0)
#   - Penalizace jsou škálované na průměrnou vzdálenost (~10 jednotek)


@dataclass(frozen=True)
class Config:
    """
    Konfigurace pro referenční MILP model
    
    Dataset: small_dataset.xlsx (abstraktní souřadnice)
    - Průměrná vzdálenost: ~10 jednotek
    - Rozsah: 0.24 - 21.06 jednotek
    """
    
    # Parametry instance
    m: int = 3                     # Počet vozidel
    Q: int = 13                    # Kapacita vozidla (unit demand => max zákazníků)
    
    # Časové parametry (ABSTRAKTNÍ JEDNOTKY)
    service_time_h: float = 1.0    # Servisní čas [časové jednotky]
                                   # Simuluje čas nakládky/vykládky
    
    speed_kmh: float = 1.0         # Rychlost [jednotky vzdálenosti / časové jednotky]
                                   # 1.0 => čas = vzdálenost (zjednodušení)
    
    # Penalizace (škálované na dataset ~10 jednotek průměr)
    pi: float = 1.5                # Penalizace za 1 jednotku zpoždění [jednotky/čas]
                                   # ~15% průměrné vzdálenosti
    
    Pskip: float = 18.0            # Penalizace za neobsloužení [jednotky]
                                   # ~1.8× průměrná vzdálenost
    
    # Solver parametry
    time_limit_sec: int = 1800     # 30 minut (reálný čas běhu solveru)
    gapRel: float = 0.10           # 10% relativní gap

# Vytvoření konfigurace
cfg = Config()

# Výpis konfigurace
print("=" * 80)
print("KONFIGURACE REFERENČNÍHO MILP MODELU")
print("=" * 80)
print(f"Dataset:          {os.path.basename(DATA_PATH)}")
print(f"Depo:             {DEPOT_CITY_NAME}")
print(f"")
print(f" POZOR: Pracujeme v ABSTRAKTNÍCH JEDNOTKÁCH (ne km, ne hodiny)!")
print(f"")
print(f"PARAMETRY INSTANCE:")
print(f"  Počet vozidel:  {cfg.m}")
print(f"  Kapacita:       {cfg.Q} zákazníků (unit demand)")
print(f"  Servisní čas:   {cfg.service_time_h} časových jednotek")
print(f"  Rychlost:       {cfg.speed_kmh} jednotek vzdálenosti/čas")
print(f"")
print

KONFIGURACE REFERENČNÍHO MILP MODELU
Dataset:          small_dataset.xlsx
Depo:             City 1

 POZOR: Pracujeme v ABSTRAKTNÍCH JEDNOTKÁCH (ne km, ne hodiny)!

PARAMETRY INSTANCE:
  Počet vozidel:  3
  Kapacita:       13 zákazníků (unit demand)
  Servisní čas:   1.0 časových jednotek
  Rychlost:       1.0 jednotek vzdálenosti/čas



<function print(*args, sep=' ', end='\n', file=None, flush=False)>

## 2. Načtení datasetu

Načtení datového souboru small_dataset.xlsx (Příloha 1) obsahujícího syntetické souřadnice 30 uzlů (Henning, 2024). Uzel City 1 je označen jako depot (index 0), zbývajících 29 uzlů tvoří množinu zákazníků C = {1, 2, …, 29} dle definice v kapitole 1.2.

In [3]:
import pandas as pd

# NAČTENÍ DATASETU small_dataset.xlsx

print("Načítání datasetu:", DATA_PATH)
print()

# Načtení surových dat
df_raw = pd.read_excel(DATA_PATH)
print("Sloupce v souboru:", list(df_raw.columns))

# Standardizace názvů sloupců (zajištění konzistence)
df_raw.columns = df_raw.columns.str.strip()

# Přejmenování sloupců na jednotný formát
if 'City' in df_raw.columns:
    df_raw = df_raw.rename(columns={'City': 'city'})
elif 'Cities' in df_raw.columns:
    df_raw = df_raw.rename(columns={'Cities': 'city'})

df_raw = df_raw.rename(columns={
    'Latitude': 'lat',
    'Longitude': 'lon'
})

# Přesuň depo (City 1) na první pozici (index 0)
depot_idx = df_raw[df_raw['city'] == DEPOT_CITY_NAME].index

if len(depot_idx) == 0:
    # Pokud není nalezeno, použij první město
    print(f"⚠ Město '{DEPOT_CITY_NAME}' nenalezeno. Použiji první město jako depo.")
    df_nodes = df_raw.reset_index(drop=True)
else:
    # Přesuň depo na první pozici
    depot_row = df_raw.loc[depot_idx[0]]
    df_others = df_raw.drop(depot_idx).reset_index(drop=True)
    df_nodes = pd.concat([pd.DataFrame([depot_row]), df_others], ignore_index=True)

# Výsledek načtení
print()
print("=" * 80)
print("NAČTENÝ DATASET")
print("=" * 80)
print(f"Celkem měst:      {len(df_nodes)}")
print(f"Depo (index 0):   {df_nodes.iloc[0]['city']}")
print(f"Zákazníci:        {len(df_nodes)-1} měst (indexy 1-{len(df_nodes)-1})")
print("=" * 80)
print()

# Zobrazení prvních 10 měst
display(df_nodes.head(10))

Načítání datasetu: small_dataset.xlsx

Sloupce v souboru: ['City', 'Latitude', 'Longitude']

NAČTENÝ DATASET
Celkem měst:      30
Depo (index 0):   City 1
Zákazníci:        29 měst (indexy 1-29)



,city,lat,lon
0,City 1,-0.935834,-1.771533
1,City 2,-9.326265,13.902797
2,City 3,-8.233092,14.873970
3,City 4,3.272331,3.961212
4,City 5,1.175829,-0.699877
5,City 6,-7.685728,16.162204
6,City 7,-0.637752,-0.173636
7,City 8,4.194633,3.705915
8,City 9,5.412912,6.754886
9,City 10,-0.861755,0.283627


## 3. Výpočet vzdálenostní matice

Výpočet symetrické matice euklidovských vzdáleností c_ij pro všechny dvojice uzlů i, j ∈ N, i ≠ j. Při jednotkové rychlosti platí t_ij = c_ij dle rovnice (9) v kapitole 1.3. Statistické vlastnosti matice jsou uvedeny v tabulce 2.1 diplomové práce.

In [4]:
import numpy as np
from math import sqrt

# VÝPOČET EUKLIDOVSKÉ VZDÁLENOSTNÍ MATICE

n = len(df_nodes)

# Vytvoření matice vzdáleností (Euklidovská metrika)
c = np.zeros((n, n))

for i in range(n):
    lat_i = df_nodes.loc[i, 'lat']
    lon_i = df_nodes.loc[i, 'lon']
    
    for j in range(n):
        if i == j:
            c[i, j] = 0.0  # Vzdálenost k sobě samému = 0
        else:
            lat_j = df_nodes.loc[j, 'lat']
            lon_j = df_nodes.loc[j, 'lon']
            
            # Euklidovská vzdálenost
            c[i, j] = sqrt((lat_i - lat_j)**2 + (lon_i - lon_j)**2)

# Výpočet cestovního času (pro abstraktní jednotky: čas = vzdálenost)
t = c / cfg.speed_kmh  # speed_kmh = 1.0 → t = c

# Statistiky vzdáleností
distances_list = []
for i in range(n):
    for j in range(i+1, n):
        distances_list.append(c[i, j])

distances_array = np.array(distances_list)

print("=" * 80)
print("VZDÁLENOSTNÍ MATICE")
print("=" * 80)
print(f"Rozměr:           {n} × {n}")
print(f"Metrika:          Euklidovská")
print()
print(f"STATISTIKY VZDÁLENOSTÍ:")
print(f"  Minimální:      {distances_array.min():.2f} jednotek")
print(f"  Maximální:      {distances_array.max():.2f} jednotek")
print(f"  Průměrná:       {distances_array.mean():.2f} jednotek")
print(f"  Medián:         {np.median(distances_array):.2f} jednotek")
print(f"  Std. odchylka:  {distances_array.std():.2f} jednotek")
print("=" * 80)

VZDÁLENOSTNÍ MATICE
Rozměr:           30 × 30
Metrika:          Euklidovská

STATISTIKY VZDÁLENOSTÍ:
  Minimální:      0.24 jednotek
  Maximální:      21.06 jednotek
  Průměrná:       10.17 jednotek
  Medián:         9.97 jednotek
  Std. odchylka:  6.54 jednotek


## 4. Generování časových oken

Deterministické generování časových oken [e_i, l_i] pro všechny zákazníky i ∈ C se seedem SEED = 42 dle parametrů v tabulce 2.2 diplomové práce. Časové okno depota je [0; 120]. Šířka oken zákazníků je proporcionální ke vzdálenosti od depota. Dolní mez e_i je tvrdým omezením (podmínka C7), horní mez l_i je měkká s penalizací za zpoždění (podmínka C8).

In [5]:
import random

# GENEROVÁNÍ ČASOVÝCH OKEN (Solomon-like)

# Nastavení seedu pro reprodukovatelnost
random.seed(42)
np.random.seed(42)

# Časové okno pro depo (vždy otevřeno)
e = np.zeros(n)
l = np.zeros(n)

# Depo: velké časové okno (celý pracovní den)
e[0] = 0.0
l[0] = 1000.0  # Dostatečně velké číslo

# ŠÍŘKA ČASOVÝCH OKEN - volitelné varianty:

# Úzká okna:     WINDOW_WIDTH_FACTOR = 0.3  (restriktivní)
# Střední okna:  WINDOW_WIDTH_FACTOR = 0.5  (DOPORUČENO - standardní)
# Široká okna:   WINDOW_WIDTH_FACTOR = 0.8  (flexibilní)

WINDOW_WIDTH_FACTOR = 0.5  # ZMĚNA PODLE POTŘEBY

print("=" * 80)
print("GENEROVÁNÍ ČASOVÝCH OKEN")
print("=" * 80)
print(f"Metoda:           Solomon-like (proporcionální k vzdálenosti)")
print(f"Typ oken:         Měkká (soft) - zpoždění povoleno s penalizací")
print(f"Šířka okna:       {WINDOW_WIDTH_FACTOR} × vzdálenost_od_depota")
print(f"")

# Zákazníci: generování založené na vzdálenosti od depota
for i in range(1, n):
    # Přímá vzdálenost od depota
    dist_from_depot = c[0, i]
    
    # Nejdříve možný příjezd (přímá cesta z depota + servisní čas v depotu)
    earliest_arrival = t[0, i] + cfg.service_time_h
    
    # Šířka časového okna (proporcionální k vzdálenosti)
    # Blízká města = širší okna (více času na obsluhu)
    # Vzdálená města = užší okna (tight schedule)
    window_width = max(3.0, dist_from_depot * WINDOW_WIDTH_FACTOR)
    
    # Náhodný posun středu okna (±20% od nejdřívějšího příjezdu)
    # Simuluje různé preference zákazníků
    center_shift = earliest_arrival * random.uniform(-0.2, 0.2)
    window_center = earliest_arrival + center_shift
    
    # Časové okno: [centrum - polovina šířky, centrum + polovina šířky]
    e[i] = max(0.0, window_center - window_width / 2)
    l[i] = window_center + window_width / 2

# Přidání časových oken do DataFrame
df_nodes['e'] = e
df_nodes['l'] = l
df_nodes['window_width'] = l - e

# Statistiky
print(f"STATISTIKY ČASOVÝCH OKEN:")
print(f"  Depo [e₀, l₀]:         [{e[0]:.1f}, {l[0]:.1f}] (vždy otevřeno)")
print(f"  ")
print(f"  Zákazníci:")
print(f"    Průměrná šířka:      {df_nodes['window_width'].iloc[1:].mean():.2f} časových jednotek")
print(f"    Min. šířka:          {df_nodes['window_width'].iloc[1:].min():.2f} časových jednotek")
print(f"    Max. šířka:          {df_nodes['window_width'].iloc[1:].max():.2f} časových jednotek")
print(f"    Std. odchylka:       {df_nodes['window_width'].iloc[1:].std():.2f} časových jednotek")
print(f"")
print(f"  Nejužší okno:          City {df_nodes['window_width'].iloc[1:].idxmin()} "
      f"(šířka: {df_nodes['window_width'].iloc[1:].min():.2f})")
print(f"  Nejširší okno:         City {df_nodes['window_width'].iloc[1:].idxmax()} "
      f"(šířka: {df_nodes['window_width'].iloc[1:].max():.2f})")
print("=" * 80)
print()

# Zobrazení prvních 10 měst s časovými okny
display(df_nodes[['city', 'lat', 'lon', 'e', 'l', 'window_width']].head(10))

GENEROVÁNÍ ČASOVÝCH OKEN
Metoda:           Solomon-like (proporcionální k vzdálenosti)
Typ oken:         Měkká (soft) - zpoždění povoleno s penalizací
Šířka okna:       0.5 × vzdálenost_od_depota

STATISTIKY ČASOVÝCH OKEN:
  Depo [e₀, l₀]:         [0.0, 1000.0] (vždy otevřeno)
  
  Zákazníci:
    Průměrná šířka:      5.67 časových jednotek
    Min. šířka:          3.00 časových jednotek
    Max. šířka:          9.58 časových jednotek
    Std. odchylka:       2.56 časových jednotek

  Nejužší okno:          City 28 (šířka: 3.00)
  Nejširší okno:         City 5 (šířka: 9.58)



,city,lat,lon,e,l,window_width
0,City 1,-0.935834,-1.771533,0.000000,1000.000000,1000.000000
1,City 2,-9.326265,13.902797,15.381367,24.270742,8.889375
2,City 3,-8.233092,14.873970,10.987961,20.075353,9.087392
3,City 4,3.272331,3.961212,5.603666,9.159401,3.555735
4,City 5,1.175829,-0.699877,1.495136,4.495136,3.000000
5,City 6,-7.685728,16.162204,17.278544,26.859514,9.580971
6,City 7,-0.637752,-0.173636,1.311030,4.311030,3.000000
7,City 8,4.194633,3.705915,7.962891,11.715361,3.752470
8,City 9,5.412912,6.754886,7.051201,12.366424,5.315223
9,City 10,-0.861755,0.283627,1.461037,4.461037,3.000000


## 5. Poptávka

Jednotková poptávka (unit demand): q_i = 1 pro všechny zákazníky i ∈ C, q_0 = 0 pro depot. Kapacita vozidla Q = 13 tak představuje maximální počet zákazníků na jedné trase. Celková kapacita flotily (3 × 13 = 39) pokrývá všech 29 zákazníků.

In [6]:
# POPTÁVKA (UNIT DEMAND)

# Unit demand: každý zákazník má poptávku = 1
q = np.ones(n)
q[0] = 0  # Depo nemá poptávku

# Přidání do DataFrame
df_nodes['demand'] = q

print("=" * 80)
print("POPTÁVKA (DEMAND)")
print("=" * 80)
print(f"Typ:              Unit demand")
print(f"Depo (index 0):   q₀ = {q[0]:.0f}")
print(f"Zákazníci:        qᵢ = {q[1]:.0f} pro všechny i ∈ {{1, ..., {n-1}}}")
print()
print(f"KAPACITA VOZIDLA:")
print(f"  Q = {cfg.Q} → max. {cfg.Q} zákazníků na vozidlo")
print(f"  {cfg.m} vozidel × {cfg.Q} kapacita = {cfg.m * cfg.Q} (pokrytí všech {n-1} zákazníků)")
print("=" * 80)

POPTÁVKA (DEMAND)
Typ:              Unit demand
Depo (index 0):   q₀ = 0
Zákazníci:        qᵢ = 1 pro všechny i ∈ {1, ..., 29}

KAPACITA VOZIDLA:
  Q = 13 → max. 13 zákazníků na vozidlo
  3 vozidel × 13 kapacita = 39 (pokrytí všech 29 zákazníků)


## 6. Definice rozhodovacích proměnných a účelové funkce

Vytvoření MILP modelu dle kompletní formulace v kapitole 1.3. Model obsahuje šest typů rozhodovacích proměnných:
- **x_ij^k** ∈ {0, 1} — trasovací proměnná (vozidlo k projíždí hranou i → j)
- **y_i^k** ∈ {0, 1} — přiřazení zákazníka i vozidlu k
- **y_i** ∈ {0, 1} — agregovaná proměnná obsloužení zákazníka i
- **s_i^k** ≥ 0 — čas zahájení obsluhy v uzlu i vozidlem k
- **T_i^k** ≥ 0 — zpoždění (tardiness) v uzlu i pro vozidlo k; v kódu označeno jako `tau`
- **p_i^k** ∈ [0, Q] — kumulativní naloženost vozidla k po opuštění uzlu i (MTZ)

Účelová funkce dle rovnice (8) minimalizuje součet cestovních nákladů, penalizací za neobsloužení (P_skip) a penalizací za zpoždění (γ · T_i^k).

In [7]:
import pulp
import time

# VYTVOŘENÍ MILP MODELU

print("=" * 80)
print("VYTVÁŘENÍ MILP MODELU")
print("=" * 80)

start_time = time.time()

# Vytvoření prázdného modelu
model = pulp.LpProblem("Orienteering_CVRPTW_Soft_TW", pulp.LpMinimize)

# Big-M hodnoty (co nejmenší pro lepší LP relaxaci)
M_time = max(l) + np.max(t) * n + cfg.service_time_h * n  # Maximální možná cesta
M_capacity = cfg.Q
M_position = n

print(f"Big-M hodnoty:")
print(f"  M_time:     {M_time:.2f}")
print(f"  M_capacity: {M_capacity}")
print(f"  M_position: {M_position}")
print()

# ROZHODOVACÍ PROMĚNNÉ

# x[k][i][j] = 1 pokud vozidlo k jede z i do j
x = {}
for k in range(cfg.m):
    x[k] = {}
    for i in range(n):
        x[k][i] = {}
        for j in range(n):
            if i == j:
                x[k][i][j] = 0  # Nelze jet z uzlu do sebe sama
            else:
                x[k][i][j] = pulp.LpVariable(
                    f"x_{k}_{i}_{j}",
                    cat=pulp.LpBinary
                )

# yik[k][i] = 1 pokud zákazník i je obsloužen vozidlem k
yik = {}
for k in range(cfg.m):
    yik[k] = {}
    for i in range(n):
        yik[k][i] = pulp.LpVariable(
            f"yik_{k}_{i}",
            cat=pulp.LpBinary
        )

# y[i] = 1 pokud zákazník i je obsloužen (agregovaná)
y = {}
for i in range(n):
    y[i] = pulp.LpVariable(
        f"y_{i}",
        cat=pulp.LpBinary
    )

# s[k][i] = čas zahájení obsluhy v uzlu i vozidlem k
s = {}
for k in range(cfg.m):
    s[k] = {}
    for i in range(n):
        s[k][i] = pulp.LpVariable(
            f"s_{k}_{i}",
            lowBound=0,
            cat=pulp.LpContinuous
        )

# tau[k][i] = zpoždění (tardiness) v uzlu i pro vozidlo k
tau = {}
for k in range(cfg.m):
    tau[k] = {}
    for i in range(n):
        tau[k][i] = pulp.LpVariable(
            f"tau_{k}_{i}",
            lowBound=0,
            cat=pulp.LpContinuous
        )

# p[k][i] = kumulativní naloženost (load propagation / MTZ-like)
p = {}
for k in range(cfg.m):
    p[k] = {}
    for i in range(n):
        p[k][i] = pulp.LpVariable(
            f"p_{k}_{i}",
            lowBound=0,
            upBound=cfg.Q,
            cat=pulp.LpContinuous
        )

print("Proměnné vytvořeny:")
print(f"  x (routing):        {cfg.m} × {n} × {n} = {cfg.m * n * (n-1)} binárních")
print(f"  yik (assignment):   {cfg.m} × {n} = {cfg.m * n} binárních")
print(f"  y (served):         {n} binárních")
print(f"  s (start time):     {cfg.m} × {n} = {cfg.m * n} spojitých")
print(f"  tau (tardiness):    {cfg.m} × {n} = {cfg.m * n} spojitých")
print(f"  p (load):           {cfg.m} × {n} = {cfg.m * n} spojitých")
print()

# ÚČELOVÁ FUNKCE

# Cestovní náklady
travel_cost = pulp.lpSum(
    c[i, j] * x[k][i][j]
    for k in range(cfg.m)
    for i in range(n)
    for j in range(n)
    if i != j
)

# Penalizace za neobsloužení
skip_cost = pulp.lpSum(
    cfg.Pskip * (1 - y[i])
    for i in range(1, n)  # Pouze zákazníci (ne depo)
)

# Penalizace za zpoždění
tardiness_cost = pulp.lpSum(
    cfg.pi * tau[k][i]
    for k in range(cfg.m)
    for i in range(1, n)  # Pouze zákazníci
)

# Celková účelová funkce
model += travel_cost + skip_cost + tardiness_cost, "Total_Cost"

print("Účelová funkce:")
print("  minimize: cestovní_náklady + penalizace_skip + penalizace_zpoždění")
print()

build_time = time.time() - start_time
print(f"✅ Model vytvořen za {build_time:.2f} sekund")
print("=" * 80)

VYTVÁŘENÍ MILP MODELU
Big-M hodnoty:
  M_time:     1661.94
  M_capacity: 13
  M_position: 30

Proměnné vytvořeny:
  x (routing):        3 × 30 × 30 = 2610 binárních
  yik (assignment):   3 × 30 = 90 binárních
  y (served):         30 binárních
  s (start time):     3 × 30 = 90 spojitých
  tau (tardiness):    3 × 30 = 90 spojitých
  p (load):           3 × 30 = 90 spojitých

Účelová funkce:
  minimize: cestovní_náklady + penalizace_skip + penalizace_zpoždění

✅ Model vytvořen za 0.11 sekund


## 7. Omezující podmínky (C1)–(C11)

Implementace jedenácti skupin omezujících podmínek dle kapitoly 1.3:
- **(C1)–(C2):** Výjezd z depota a návrat do depota — každé vozidlo vyjede a vrátí se právě jednou
- **(C3)–(C4):** Kontinuita toku — pokud vozidlo k obslouží zákazníka i, musí do uzlu přijet i z něj odjet
- **(C5):** Orienteering princip — každý zákazník je obsloužen nejvýše jedním vozidlem
- **(C6):** Časová propagace s Big-M formulací zajišťující správnou posloupnost návštěv na trase
- **(C7):** Tvrdá dolní mez časového okna (s_i^k ≥ e_i)
- **(C8):** Měkká horní mez časového okna (s_i^k ≤ l_i + T_i^k)
- **(C9):** Propagace kumulativní naloženosti (MTZ formulace pro eliminaci podcyklů)
- **(C10):** Kapacitní omezení (p_i^k ≤ Q)
- **(C11):** Zákaz smyček (x_ii^k = 0, zajištěno implicitně v definici proměnných)

In [8]:
# OMEZENÍ (CONSTRAINTS)

print("=" * 80)
print("PŘIDÁVÁNÍ OMEZENÍ")
print("=" * 80)

constraint_start = time.time()

# C1: Výjezd z depota - každé vozidlo vyjede právě jednou

for k in range(cfg.m):
    model += (
        pulp.lpSum(x[k][0][j] for j in range(1, n)) == 1,
        f"depot_outbound_{k}"
    )

print(f"✓ C1: Výjezd z depota ({cfg.m} omezení)")

# C2: Návrat do depota - každé vozidlo se vrátí právě jednou

for k in range(cfg.m):
    model += (
        pulp.lpSum(x[k][i][0] for i in range(1, n)) == 1,
        f"depot_inbound_{k}"
    )

print(f"✓ C2: Návrat do depota ({cfg.m} omezení)")

# C3: Kontinuita trasy - výstup (pokud vozidlo obslouží, musí odjet)

for k in range(cfg.m):
    for i in range(1, n):
        model += (
            pulp.lpSum(x[k][i][j] for j in range(n) if j != i) == yik[k][i],
            f"flow_out_{k}_{i}"
        )

print(f"✓ C3: Kontinuita trasy - výstup ({cfg.m * (n-1)} omezení)")

# C4: Kontinuita trasy - vstup (pokud vozidlo obslouží, musí přijet)

for k in range(cfg.m):
    for i in range(1, n):
        model += (
            pulp.lpSum(x[k][j][i] for j in range(n) if j != i) == yik[k][i],
            f"flow_in_{k}_{i}"
        )

print(f"✓ C4: Kontinuita trasy - vstup ({cfg.m * (n-1)} omezení)")

# C5: Orienteering princip - každý zákazník max. 1× obsloužen

for i in range(1, n):
    # Agregace přes všechna vozidla
    model += (
        pulp.lpSum(yik[k][i] for k in range(cfg.m)) == y[i],
        f"aggregation_{i}"
    )
    
    # Maximálně jednou obsloužen
    model += (
        y[i] <= 1,
        f"orienteering_{i}"
    )

print(f"✓ C5: Orienteering princip ({2 * (n-1)} omezení)")

# C6: Časová propagace - návaznost časů na trase

for k in range(cfg.m):
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            
            model += (
                s[k][j] >= s[k][i] + cfg.service_time_h + t[i, j] 
                         - M_time * (1 - x[k][i][j]),
                f"time_prop_{k}_{i}_{j}"
            )

print(f"✓ C6: Časová propagace ({cfg.m * n * (n-1)} omezení)")

# C7: Dolní mez časového okna

for k in range(cfg.m):
    for i in range(1, n):
        model += (
            s[k][i] >= e[i] - M_time * (1 - yik[k][i]),
            f"tw_lower_{k}_{i}"
        )

print(f"✓ C7: Dolní mez TW ({cfg.m * (n-1)} omezení)")


# C8: Horní mez časového okna (soft - s tardiness)

for k in range(cfg.m):
    for i in range(1, n):
        model += (
            s[k][i] <= l[i] + tau[k][i] + M_time * (1 - yik[k][i]),
            f"tw_upper_soft_{k}_{i}"
        )

print(f"✓ C8: Horní mez TW (soft) ({cfg.m * (n-1)} omezení)")


# C9: Load propagation (kumulativní naloženost + subtour elimination)

for k in range(cfg.m):
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            
            model += (
                p[k][j] >= p[k][i] + q[j] - M_capacity * (1 - x[k][i][j]),
                f"load_prop_{k}_{i}_{j}"
            )

print(f"✓ C9: Load propagation ({cfg.m * n * (n-1)} omezení)")

# C10: Kapacitní omezení

for k in range(cfg.m):
    for i in range(n):
        model += (
            p[k][i] <= cfg.Q,
            f"capacity_{k}_{i}"
        )

print(f"✓ C10: Kapacitní omezení ({cfg.m * n} omezení)")


# C11: Zakázané smyčky (self-loops)

# Již zajištěno v definici x (x[k][i][i] = 0)
print(f"✓ C11: Zakázané smyčky (implicitní)")

# Celkový čas přidání omezení
constraint_time = time.time() - constraint_start

# Souhrn
total_constraints = len(model.constraints)
total_variables = len(model.variables())

print()
print("=" * 80)
print("SOUHRN MODELU")
print("=" * 80)
print(f"Proměnné celkem:  {total_variables}")
print(f"  - Binární:      {cfg.m * n * (n-1) + cfg.m * n + n}")
print(f"  - Spojité:      {3 * cfg.m * n}")
print()
print(f"Omezení celkem:   {total_constraints}")
print()
print(f"Čas vytvoření:    {build_time:.2f}s")
print(f"Čas omezení:      {constraint_time:.2f}s")
print(f"Celkem:           {build_time + constraint_time:.2f}s")
print("=" * 80)

PŘIDÁVÁNÍ OMEZENÍ
✓ C1: Výjezd z depota (3 omezení)
✓ C2: Návrat do depota (3 omezení)
✓ C3: Kontinuita trasy - výstup (87 omezení)
✓ C4: Kontinuita trasy - vstup (87 omezení)
✓ C5: Orienteering princip (58 omezení)
✓ C6: Časová propagace (2610 omezení)
✓ C7: Dolní mez TW (87 omezení)
✓ C8: Horní mez TW (soft) (87 omezení)
✓ C9: Load propagation (2610 omezení)
✓ C10: Kapacitní omezení (90 omezení)
✓ C11: Zakázané smyčky (implicitní)

SOUHRN MODELU
Proměnné celkem:  2993
  - Binární:      2730
  - Spojité:      270

Omezení celkem:   5722

Čas vytvoření:    0.11s
Čas omezení:      0.37s
Celkem:           0.47s


## 8. Řešení modelu solverem CBC

Spuštění open-source solveru CBC s konfigurací dle tabulky 2.7 diplomové práce: časový limit 1 800 s, relativní toleranční gap 10 %, automatický počet vláken. Solver využívá presolve, řezy a heuristiky.

In [9]:
# ŘEŠENÍ MILP MODELU

print()
print("=" * 80)
print("SPOUŠTĚNÍ SOLVERU")
print("=" * 80)
print(f"Solver:           CBC (open-source)")
print(f"Time limit:       {cfg.time_limit_sec}s ({cfg.time_limit_sec/60:.0f} min)")
print(f"Gap tolerance:    {cfg.gapRel*100:.0f}%")
print(f"Threads:          auto (všechny dostupné)")
print("=" * 80)
print()
print("Řešení probíhá...")
print()

# Nastavení solveru
solver = pulp.PULP_CBC_CMD(
    timeLimit=cfg.time_limit_sec,
    gapRel=cfg.gapRel,
    threads=None,  # Automaticky všechna jádra
    msg=1,         # Zobrazit výstup solveru
    options=['presolve on', 'cuts on', 'heuristics on']
)

# Spuštění řešení
solve_start = time.time()
model.solve(solver)
solve_time = time.time() - solve_start

# Výsledky
status = pulp.LpStatus[model.status]
obj = pulp.value(model.objective)

print()
print("=" * 80)
print("ŘEŠENÍ DOKONČENO")
print("=" * 80)
print(f"Runtime:          {solve_time:.2f}s ({solve_time/60:.1f} min)")
print(f"Status:           {status}")
print(f"Objective:        {obj:.2f} jednotek" if obj is not None else "Objective:        None")
print("=" * 80)


SPOUŠTĚNÍ SOLVERU
Solver:           CBC (open-source)
Time limit:       1800s (30 min)
Gap tolerance:    10%
Threads:          auto (všechny dostupné)

Řešení probíhá...


ŘEŠENÍ DOKONČENO
Runtime:          1799.74s (30.0 min)
Status:           Not Solved
Objective:        28.60 jednotek


## 9. Analýza výsledků

Rekalkulace hodnoty účelové funkce z rozhodovacích proměnných a audit celočíselnosti řešení. Výstupem jsou klíčové metriky dle tabulky 2.5 diplomové práce: hodnota účelové funkce Z(R), jednotlivé složky dle rovnice (8) a počet obsloužených zákazníků. Audit ověřuje, zda trasovací proměnné x_ij^k nabývají celočíselných hodnot, nebo zda solver vrátil pouze LP relaxaci.

In [10]:
# ANALÝZA VÝSLEDKŮ

print()
print("=" * 80)
print("ANALÝZA ŘEŠENÍ")
print("=" * 80)

# Recomputation z proměnných (striktní)
travel_km_from_x = 0.0
for k in range(cfg.m):
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            v = pulp.value(x[k][i][j])
            if v is not None:
                travel_km_from_x += float(c[i, j]) * float(v)

skip_cost_from_y = 0.0
served_count_relaxed = 0.0
for i in range(1, n):
    yi = pulp.value(y[i])
    if yi is not None:
        served_count_relaxed += float(yi)
        skip_cost_from_y += float(cfg.Pskip) * (1.0 - float(yi))

tard_h_from_tau = 0.0
for k in range(cfg.m):
    for i in range(1, n):
        tv = pulp.value(tau[k][i])
        if tv is not None:
            tard_h_from_tau += float(tv)

tard_cost_from_tau = float(cfg.pi) * tard_h_from_tau
objective_recomputed = travel_km_from_x + skip_cost_from_y + tard_cost_from_tau

# Audit integrálnosti
eps = 1e-6
count_frac = 0
for k in range(cfg.m):
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            v = pulp.value(x[k][i][j])
            if v is not None and eps < v < 1 - eps:
                count_frac += 1

is_integral = (count_frac == 0)

# Výpis KPI
print()
print("KLÍČOVÉ UKAZATELE (KPI):")
print(f"  Objective (model):        {obj:.2f}" if obj is not None else "  Objective (model):        None")
print(f"  Objective (recomputed):   {objective_recomputed:.2f}")
print()
print(f"  Cestovní náklady:         {travel_km_from_x:.2f} jednotek")
print(f"  Skip cost:                {skip_cost_from_y:.2f} jednotek")
print(f"  Tardiness cost:           {tard_cost_from_tau:.2f} jednotek")
print()
print(f"  Zpoždění celkem:          {tard_h_from_tau:.2f} časových jednotek")
print(f"  Obslouženo zákazníků:     {served_count_relaxed:.2f}")
print(f"  Neobslouženo:             {n - 1 - served_count_relaxed:.2f}")
print()
print(f"KVALITA ŘEŠENÍ:")
print(f"  Fractionální hrany (x):   {count_frac}")
print(f"  Je integrální:            {is_integral}")

if is_integral:
    print(f" ŘEŠENÍ JE VALIDNÍ (binární proměnné)")
else:
    print(f" ŘEŠENÍ JE FRACTIONÁLNÍ (LP relaxace, ne MILP)")

print("=" * 80)


ANALÝZA ŘEŠENÍ

KLÍČOVÉ UKAZATELE (KPI):
  Objective (model):        28.60
  Objective (recomputed):   28.60

  Cestovní náklady:         28.60 jednotek
  Skip cost:                0.00 jednotek
  Tardiness cost:           0.00 jednotek

  Zpoždění celkem:          0.00 časových jednotek
  Obslouženo zákazníků:     29.00
  Neobslouženo:             0.00

KVALITA ŘEŠENÍ:
  Fractionální hrany (x):   64
  Je integrální:            False
 ŘEŠENÍ JE FRACTIONÁLNÍ (LP relaxace, ne MILP)


## 10. Rekonstrukce tras

Extrakce tras jednotlivých vozidel z binárních proměnných x_ij^k. Pro každé vozidlo k je rekonstruována posloupnost uzlů od depota zpět do depota na základě proměnných s hodnotou x_ij^k > 0,5. Rekonstrukce je provedena pouze v případě celočíselného řešení; při LP relaxaci trasy nelze jednoznačně určit.

In [11]:
# REKONSTRUKCE TRAS

def extract_routes_integral(x, n, m, threshold=0.5):
    """
    Extrakce tras z binárních proměnných x
    """
    routes = []
    
    for k in range(m):
        # Vytvoř následníka (successor) pro každý uzel
        succ = {}
        for i in range(n):
            for j in range(n):
                if i == j:
                    continue
                v = pulp.value(x[k][i][j])
                if v is not None and v > threshold:
                    succ[i] = j
        
        # Rekonstrukce trasy od depota
        route = [0]
        cur = 0
        visited = {0}
        
        while cur in succ:
            nxt = succ[cur]
            route.append(nxt)
            
            if nxt == 0:  # Návrat do depota
                break
            
            if nxt in visited:  # Subtour (nemělo by nastat)
                break
            
            visited.add(nxt)
            cur = nxt
        
        routes.append(route)
    
    return routes

print()
print("=" * 80)
print("TRASY VOZIDEL")
print("=" * 80)

if is_integral:
    routes = extract_routes_integral(x, n=n, m=cfg.m, threshold=0.5)
    
    for k, route in enumerate(routes, start=1):
        # Názvy měst
        city_names = [df_nodes.loc[i, 'city'] for i in route]
        city_route = " → ".join(city_names)
        
        # Délka trasy
        route_dist = sum(c[route[i], route[i+1]] for i in range(len(route)-1))
        
        # Počet zákazníků
        num_customers = len([r for r in route if r != 0])
        
        print(f"\nVozidlo {k}:")
        print(f"  Trasa:          {city_route}")
        print(f"  Zákazníků:      {num_customers}")
        print(f"  Délka:          {route_dist:.2f} jednotek")
        
        # Časy návštěv
        print(f"  Časy:")
        for i in route:
            s_val = pulp.value(s[k-1][i])
            if s_val is not None:
                print(f"    {df_nodes.loc[i, 'city']:15s}: t={s_val:6.2f}")
    
else:
    print(" Řešení není integrální - trasy nelze rekonstruovat.")
    print("   (Pravděpodobně kvůli time limitu - solver nenašel feasible integer solution)")
    routes = None

print("=" * 80)


TRASY VOZIDEL
 Řešení není integrální - trasy nelze rekonstruovat.
   (Pravděpodobně kvůli time limitu - solver nenašel feasible integer solution)


## 11. Souhrnná tabulka výsledků

Agregace výsledků do jednotného formátu pro srovnání s heuristickými metodami (kapitola 2.5). Tabulka obsahuje konfiguraci instance, metriky dle tabulky 2.5 a informaci o celočíselnosti řešení.

In [12]:
# VÝSLEDNÁ TABULKA PRO BENCHMARKING

result_row = {
    "method": "MILP_CBC",
    "dataset": "small_dataset.xlsx",
    "n_cities": n,
    "m_vehicles": cfg.m,
    "capacity_Q": cfg.Q,
    "service_time": cfg.service_time_h,
    "P_skip": cfg.Pskip,
    "pi": cfg.pi,
    "time_limit_sec": cfg.time_limit_sec,
    "gap_tolerance": cfg.gapRel,
    
    "runtime_sec": round(solve_time, 2),
    "status": status,
    
    "objective_model": round(obj, 2) if obj is not None else None,
    "objective_recomputed": round(objective_recomputed, 2),
    
    "travel_cost": round(travel_km_from_x, 2),
    "skip_cost": round(skip_cost_from_y, 2),
    "tardiness_cost": round(tard_cost_from_tau, 2),
    "tardiness_hours": round(tard_h_from_tau, 2),
    
    "served_count": round(served_count_relaxed, 2),
    "skipped_count": round(n - 1 - served_count_relaxed, 2),
    
    "x_fractional_arcs": count_frac,
    "is_integral": is_integral,
}

# Zobrazení jako DataFrame
df_result = pd.DataFrame([result_row])

print()
print("=" * 80)
print("VÝSLEDNÁ TABULKA")
print("=" * 80)
display(df_result)
print()



VÝSLEDNÁ TABULKA


,method,dataset,n_cities,m_vehicles,capacity_Q,service_time,P_skip,pi,time_limit_sec,gap_tolerance,...,objective_model,objective_recomputed,travel_cost,skip_cost,tardiness_cost,tardiness_hours,served_count,skipped_count,x_fractional_arcs,is_integral
0,MILP_CBC,small_dataset.xlsx,30,3,13,1.0,18.0,1.5,1800,0.1,...,28.6,28.6,28.6,0.0,0.0,0.0,29.0,0.0,64,False
